# 05. RoBERTa Fine-Tuning

### Approach:
- Load pre-trained RoBERTa model (`roberta-base` or `cardiffnlp/twitter-roberta-base-sentiment-latest`).
- Attach dynamic sequence classification head matching the project's classes.
- Fine-tune transformer weights with low learning rate (`2e-5`) and linear warmup.
- Evaluate on identical test set.

In [ ]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath('..'))
from src.preprocessing.nlp_pipeline import DatasetValidator, LabelEncoder
from src.models.roberta import RoBERTaClassifier
from src.evaluation.metrics import evaluate_predictions

# Load data
df = pd.read_csv('../data/sample_sentiment_benchmark.csv')
train_df, val_df, test_df = DatasetValidator.prepare_splits(df, 'text', 'sentiment')

le = LabelEncoder()
le.fit(train_df['sentiment'].tolist())
id2label = {i: lbl for i, lbl in enumerate(le.classes_)}
label2id = {lbl: i for i, lbl in enumerate(le.classes_)}

print("Configured Classes:", id2label)

In [ ]:
# Initialize RoBERTa Wrapper
try:
    roberta = RoBERTaClassifier(
        model_name_or_path="cardiffnlp/twitter-roberta-base-sentiment-latest",
        num_classes=len(le),
        id2label=id2label,
        label2id=label2id
    )
    print("Pretrained RoBERTa loaded successfully!")
except Exception as e:
    print("HuggingFace download note (offline or network fallback):", e)

In [ ]:
# Sample Inference with Confidence Calibration
sample_text = "The product is amazing but delivery was late."
try:
    pred = roberta.predict(sample_text)
    print("Prediction:", pred)
except Exception as e:
    print("Inference skipped:", e)